# CD8 T–B Synapse Axis vs SemanticSCVI Factors

Build a per-cell **T–B immunological-synapse** score from synapse-marker **abundance** + pairwise **colocalization**, and test how the 10 SemanticSCVI latent factors (from `02_semantic_factor_interpretation.ipynb`) relate to it.

**Hypothesis:** Blinatumomab / 48h / NALM-6 co-culture cells score higher on the synapse axis than Mock / 6h / healthy-B cells. A Z-factor that correlates with this axis is interpretable as a "synapse-engagement" factor.

| Cell | Purpose |
|------|---------|
| 1 | Setup — load adata, load cached SemanticSCVI, extract `Z_arr`, `W_arr` |
| 2 | Config — T/B synapse marker lists, feature-block toggle |
| 3 | Build per-cell synapse feature matrix (abundance + within-set coloc + T×B cross) |
| 4 | PCA on features + baseline scores (signature, cross-coloc sum) |
| 5 | Loadings — top features driving PC1 / PC2 |
| 6 | Metadata association (time, condition, cell_system, pseudotime) |
| 7 | 2D scatter grid: (PC1, PC2) colored by metadata |
| 8 | Heatmap: 10 Z factors × {PC1, PC2, PC3, sig, cross} (Spearman + BH-FDR) |
| 9 | Deep-dive on top Z-factor: Z-vs-PC1 scatter + side-by-side top features |
| 10 | Optional: quadrant DE (best Z factor × synapse_PC1) |

In [ ]:
# Cell 1 — Setup, Load Data, Validate Embeddings, Load Model, Health Check
import os, sys
for k in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[k] = '1'

import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import torch
import pickle
import matplotlib.pyplot as plt
from pathlib import Path
from scvi.model._semantic_scvi import SemanticSCVI

# --- Paths ---
_PROJECT_DIR = Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data')
_FACTOR_DIR  = _PROJECT_DIR / 'factor_analysis'
sys.path.insert(0, str(_FACTOR_DIR))
sys.path.insert(0, str(_PROJECT_DIR))

import importlib, factor_utils; importlib.reload(factor_utils)
from factor_utils import (
    CACHE_DIR, CD8_ADATA_PATH,
    configure_mpl, plot_embedding_distances,
    filter_non_t_markers, prepare_adata_for_model, extract_model_outputs, plot_training_health,
)

configure_mpl()

# ======================== CONFIGURATION ========================
SEED = 42
scvi.settings.seed = SEED
RESULTS_DIR = _FACTOR_DIR / 'results_semantic'
RESULTS_DIR.mkdir(exist_ok=True)
SEMANTIC_MODEL_DIR = CACHE_DIR / 'semantic_scvi_cd8_model'

N_TOP   = 80
N_LATENT = 10
FORCE_RETRAIN = False           # use cached model from 02_semantic_factor_interpretation

# --- Model architecture (must match the cached model) ---
N_HIDDEN               = 128
GENE_LIKELIHOOD        = 'nb'
COHERENCE_WEIGHT       = 2000.0
LOSS_MODE              = 'geometric'
N_GENE_SAMPLE          = None
USE_DECODER_BATCH_NORM = False
DECORRELATION_LOSS_WEIGHT = 120.0
Z_DECOR_WEIGHT            = 50.0

# --- Training (unused when FORCE_RETRAIN=False) ---
MAX_EPOCHS              = 500
WARMUP_EPOCHS           = 30
WARMUP_SCHEDULE         = 'cosine'
BATCH_SIZE              = 128
TRAIN_SIZE              = 0.9
EARLY_STOPPING          = True
EARLY_STOPPING_PATIENCE = 30
CHECK_VAL_EVERY_N_EPOCH = 5

# --- Non-T-cell marker filter (must match 02 notebook so var_names align with cached model) ---
REMOVE_NON_T_MARKERS = True
NON_T_CELL_MARKERS = {
    'B_cell':           ['CD19','CD20','CD21','CD22','CD72','CD180','CD79a','IgM','IgD','CD268','CD10','CD24'],
    'Plasma_cell':      ['CD138','CD269','CD319'],
    'Monocyte_macro':   ['CD14','CD64','CD163','CD206','CD13','CD33',],
    'DC':               ['CD1a','CD1b','CD1c','CD141','CD209','CD371','CD169'],
    'Granulocyte':      ['CD193','CD89','CD66b'],
    'NK_restricted':    ['NKp80','CD335','CD337',],
    'Mast_basophil':    ['CD117','IgE'],
    'Platelet_endoth':  ['CD41','CD62P','CD31'],
    'Stromal_epithel':  ['CD326',],
    'Progenitor':       ['CD34'],
    'APC_costim':       ['CD40','CD80','CD86','CD35','CD37','CD123'],
}
# ===============================================================

print(f'scvi-tools: {scvi.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device: {torch.cuda.get_device_name(0)}')

# --- Load data (preserve full adata for the synapse feature build; model uses filtered copy) ---
adata_full = sc.read_h5ad(CD8_ADATA_PATH)
print(f'\nLoaded {adata_full.n_obs:,} cells x {adata_full.n_vars} markers (full)')
print(f'Layers: {list(adata_full.layers.keys())}')
print(f'obsm keys: {list(adata_full.obsm.keys())}')

emb_path = _FACTOR_DIR / 'functional_protein_embeddings.pkl'
with open(emb_path, 'rb') as f:
    emb_data = pickle.load(f)
print(f'Embeddings: {emb_data["embeddings"].shape} ({len(emb_data["dimensions"])} dims)')

plot_embedding_distances(emb_data, results_dir=RESULTS_DIR)

# --- Prepare a SEPARATE copy for the model (filter + top-N), keep adata_full for synapse axis ---
adata = adata_full.copy()
if REMOVE_NON_T_MARKERS:
    adata = filter_non_t_markers(adata, NON_T_CELL_MARKERS)
adata, semantic_map = prepare_adata_for_model(adata, emb_data, n_top=N_TOP)

# --- Load SemanticSCVI ---
SemanticSCVI.setup_anndata(adata, layer='counts', batch_key=None)
n_gene_sample = N_GENE_SAMPLE or min(1024, adata.n_vars)
model_kwargs = dict(
    semantic_map=semantic_map.float(),
    n_latent=N_LATENT, n_hidden=N_HIDDEN,
    gene_likelihood=GENE_LIKELIHOOD, coherence_weight=COHERENCE_WEIGHT,
    loss_mode=LOSS_MODE, n_gene_sample=n_gene_sample,
    use_decoder_batch_norm=USE_DECODER_BATCH_NORM,
    decorrelation_loss_weight=DECORRELATION_LOSS_WEIGHT,
    z_decor_weight=Z_DECOR_WEIGHT,
)

if SEMANTIC_MODEL_DIR.exists() and not FORCE_RETRAIN:
    print(f'Loading cached SemanticSCVI from {SEMANTIC_MODEL_DIR}')
    model = SemanticSCVI(adata, **model_kwargs)
    model_state = torch.load(SEMANTIC_MODEL_DIR / 'model.pt', map_location='cpu', weights_only=False)
    state_dict = {k: v for k, v in model_state['model_state_dict'].items()
                  if k in set(model.module.state_dict().keys())}
    model.module.load_state_dict(state_dict)
    model.is_trained_ = True
    print('Loaded cached model.')
else:
    print('Training SemanticSCVI from scratch...')
    model = SemanticSCVI(adata, **model_kwargs)
    model.train(
        max_epochs=MAX_EPOCHS, warmup_epochs=WARMUP_EPOCHS,
        warmup_schedule=WARMUP_SCHEDULE, batch_size=BATCH_SIZE,
        train_size=TRAIN_SIZE, early_stopping=EARLY_STOPPING,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        check_val_every_n_epoch=CHECK_VAL_EVERY_N_EPOCH,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    )
    SEMANTIC_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    model.save(SEMANTIC_MODEL_DIR, overwrite=True)

# --- Extract latents & weights ---
Z_arr, W_arr, W_df, marker_names, z_var = extract_model_outputs(model, adata, N_LATENT)
n_factors = N_LATENT
print(f'\nZ_arr: {Z_arr.shape}   W_arr: {W_arr.shape}   n_factors: {n_factors}')
print(f'Results dir: {RESULTS_DIR}')

In [ ]:
# Cell 2 — Synapse-axis configuration
# >>> EDIT synapse markers / blocks here <<<
T_SYNAPSE_MARKERS = ['CD3e', 'TCRab', 'CD28', 'CD2', 'CD11a', 'CD18']
B_SYNAPSE_MARKERS = ['CD19', 'HLA-ABC', 'HLA-DR-DP-DQ', 'CD86', 'CD58', 'CD54']

ABUNDANCE_OBSM = 'arcsinh'          # obsm key for per-marker abundance (rows = cells, cols = markers)
COLOC_OBSM     = 'spatial_asinh5'   # obsm key for pairwise coloc (columns named 'M1/M2')

# Which feature blocks enter the PCA. Default per plan: abundance + within-set coloc.
FEATURE_BLOCKS = ('abundance', 'within_coloc')

# PCA — auto-pick #components so cumulative explained variance >= threshold
PC_VARIANCE_THRESHOLD = 0.80         # 0.0–1.0; smallest k with cumsum(var_ratio) >= this
PC_MAX                = 50           # hard cap on PCs to fit for the scree
N_TOP_PCS             = 5            # PCs kept in synapse_df and reported downstream

# --- Validate markers exist in adata_full ---
present_T = [m for m in T_SYNAPSE_MARKERS if m in adata_full.var_names]
present_B = [m for m in B_SYNAPSE_MARKERS if m in adata_full.var_names]
missing_T = [m for m in T_SYNAPSE_MARKERS if m not in adata_full.var_names]
missing_B = [m for m in B_SYNAPSE_MARKERS if m not in adata_full.var_names]

print(f"T-side markers present: {len(present_T)}/{len(T_SYNAPSE_MARKERS)}")
print(f"B-side markers present: {len(present_B)}/{len(B_SYNAPSE_MARKERS)}")
if missing_T: print(f"  T missing: {missing_T}")
if missing_B: print(f"  B missing: {missing_B}")

# Use the filtered lists going forward
T_SYNAPSE_MARKERS = present_T
B_SYNAPSE_MARKERS = present_B


In [ ]:
# Cell 3 — Build per-cell synapse feature matrix
# Uses adata_full (all 159 markers + spatial obsm intact).
from factor_utils import build_synapse_features

features, missing = build_synapse_features(
    adata_full,
    t_markers=T_SYNAPSE_MARKERS,
    b_markers=B_SYNAPSE_MARKERS,
    abundance_obsm=ABUNDANCE_OBSM,
    coloc_obsm=COLOC_OBSM,
    blocks=FEATURE_BLOCKS,
)

abund_cols = [c for c in features.columns if c.startswith('abund:')]
coloc_cols = [c for c in features.columns if c.startswith('coloc:')]

print(f"feature matrix: {features.shape[0]} cells × {features.shape[1]} features")
print(f"  abundance:    {len(abund_cols)}")
print(f"  within-coloc: {len(coloc_cols)}   (missing pairs: {len(missing['coloc'])})")
if missing['coloc']:
    print(f"  missing within-coloc examples: {missing['coloc'][:8]}")

features.head(3)


In [ ]:
# Cell 4 — PCA on [abundance + within-coloc]
import importlib, factor_utils; importlib.reload(factor_utils)
from factor_utils import (select_pca_columns, run_synapse_pca,
                          build_synapse_score_df, orient_pc1_by_metadata,
                          plot_synapse_scree)

pca_cols = select_pca_columns(features, FEATURE_BLOCKS)
print(f"PCA input: {features.shape[0]} × {len(pca_cols)}")

pca, Z_pcs, cumvar, k_auto, n_fit = run_synapse_pca(
    features, pca_cols,
    pc_max=PC_MAX, pc_variance_threshold=PC_VARIANCE_THRESHOLD, seed=SEED,
)
if cumvar[-1] < PC_VARIANCE_THRESHOLD:
    print(f"[warn] cumulative variance at k={n_fit} is {cumvar[-1]:.3f}, "
          f"below threshold {PC_VARIANCE_THRESHOLD:.2f}")

print(f"Explained variance (PC1..PC{N_TOP_PCS}): "
      f"{np.round(pca.explained_variance_ratio_[:N_TOP_PCS], 4)}")
print(f"Cumulative         (PC1..PC{N_TOP_PCS}): "
      f"{np.round(cumvar[:N_TOP_PCS], 4)}")
print(f"k_auto @ cumvar≥{PC_VARIANCE_THRESHOLD:.2f}: {k_auto}   "
      f"(PC1..PC{k_auto} explain {cumvar[k_auto-1]*100:.1f}%)")

synapse_df = build_synapse_score_df(features, Z_pcs, n_top=N_TOP_PCS)
orient_pc1_by_metadata(synapse_df, Z_pcs, pca, adata_full)

plot_synapse_scree(pca, cumvar, k_auto, PC_VARIANCE_THRESHOLD,
                   n_features=len(pca_cols),
                   save_path=RESULTS_DIR / 'synapse_scree.png')

if {'condition','time'} <= set(adata_full.obs.columns):
    ct = pd.DataFrame({'PC1': synapse_df['synapse_PC1'].values,
                       'condition': adata_full.obs['condition'].values,
                       'time': adata_full.obs['time'].astype(str).values})
    print("\nMean synapse_PC1 by (condition, time):")
    print(ct.groupby(['condition','time'])['PC1'].mean().round(3))

synapse_df.describe().T


In [ ]:
# Cell 6 — Combined heatmap: synapse PC × [metadata + Z factors]
# Rows: synapse PCs (from synapse_df). Columns: metadata effects + Z[0]..Z[n_factors-1].
# This is the "synapse PCs as factors" view — direct analog of Cell 6+7 in notebook 02.
import importlib, factor_utils; importlib.reload(factor_utils)
from factor_utils import (compute_metadata_associations,
                          factor_program_correlation,
                          plot_correlation_heatmap)

# Align synapse_df rows to the model's adata ordering (Z_arr row order).
assert adata.n_obs == adata_full.n_obs, \
    f"cell-count mismatch: adata={adata.n_obs}, adata_full={adata_full.n_obs}"
synapse_df_aligned = synapse_df.loc[adata.obs_names]
synapse_Z = synapse_df_aligned.values.astype(float)
synapse_var = np.var(synapse_Z, axis=0)

# 1. Metadata associations for each synapse PC (rows=PCs, cols=metadata tests).
syn_assoc = compute_metadata_associations(synapse_Z, adata, z_var=synapse_var)
syn_assoc.index = synapse_df_aligned.columns

# 2. Correlation of each synapse PC with every Z factor (rows=PCs, cols=Z[0..N-1]).
Z_df = pd.DataFrame(Z_arr,
                    index=adata.obs_names,
                    columns=[f"Z[{f}]" for f in range(Z_arr.shape[1])])
syn_z_corr, syn_z_pval, syn_z_padj = factor_program_correlation(
    synapse_Z, Z_df, method='spearman',
)
syn_z_corr.index = synapse_df_aligned.columns
syn_z_padj.index = synapse_df_aligned.columns

print("Synapse PC × Z-factor Spearman ρ:")
display(syn_z_corr.round(3))
print("\nBH-adjusted p:")
display(syn_z_padj.round(4))

# 3. Single combined heatmap: rows = PCs, columns = metadata effects + Z factors.
plot_correlation_heatmap(
    assoc_df=syn_assoc,
    corr_df=syn_z_corr,
    padj_df=syn_z_padj,
    results_dir=RESULTS_DIR,
    row_labels=list(synapse_df_aligned.columns),
    title="Synapse PC × [metadata + Z factors]",
)
import shutil
src = RESULTS_DIR / 'correlation_heatmap.png'
if src.exists():
    shutil.move(str(src), str(RESULTS_DIR / 'synapse_combined_heatmap.png'))
    print(f"Saved: {RESULTS_DIR / 'synapse_combined_heatmap.png'}")

# 4. Cross-check: Z-factor × metadata association — should match notebook 02 exactly.
# (uses identical call: compute_metadata_associations(Z_arr, adata, z_var))
z_assoc = compute_metadata_associations(Z_arr, adata, z_var=z_var)
print("\n--- Cross-check: Z-factor × metadata (should match notebook 02) ---")
display(z_assoc.drop(columns=['factor']).round(3))

# 5. Top Z factor per synapse PC.
top_hits = (syn_z_corr.abs().stack()
            .reset_index()
            .rename(columns={'level_0': 'synapse_PC', 'level_1': 'Z_factor', 0: 'abs_rho'})
            .sort_values('abs_rho', ascending=False)
            .groupby('synapse_PC').head(1))
print("\nTop-correlated Z factor per synapse PC:")
print(top_hits.to_string(index=False))


In [ ]:
# Cell 5 — PC loadings for PC1..PC{N} (top 20 per PC)
import importlib, factor_utils; importlib.reload(factor_utils)
from factor_utils import plot_synapse_pc_loadings

pcs_plot = tuple(f'PC{i + 1}' for i in range(N_TOP_PCS))
loadings = plot_synapse_pc_loadings(
    pca, pca_cols, pcs=pcs_plot, n_top=20,
    save_path=RESULTS_DIR / 'synapse_pc_loadings.png',
)

for pc in pcs_plot:
    top_idx = loadings[pc].abs().sort_values(ascending=False).head(10).index
    signed  = loadings[pc].loc[top_idx]
    print(f"\n{pc} top-10 (signed, ordered by |loading|):")
    for feat, val in signed.items():
        sign = '+' if val >= 0 else '-'
        print(f"  {sign} {abs(val):.3f}   {feat}")


In [ ]:
# Cell 7 — 2D scatter grid: (synapse_PC1, synapse_PC2) colored by metadata
import importlib, factor_utils; importlib.reload(factor_utils)
from factor_utils import plot_synapse_pc_scatter_grid

plot_synapse_pc_scatter_grid(
    synapse_df, adata_full,
    pc_x='synapse_PC1', pc_y='synapse_PC2',
    meta_cols=('condition', 'time', 'cell_system', 'dpt_pseudotime'),
    save_path=RESULTS_DIR / 'synapse_pc_scatter_grid.png',
)


In [ ]:
# Cell 7c — UMAP of top synapse PCs, colored by metadata + per-PC loadings
import importlib, factor_utils; importlib.reload(factor_utils)
from factor_utils import plot_synapse_umap

N_PCS_FOR_UMAP = int(min(10, Z_pcs.shape[1]))
print(f"UMAP on first {N_PCS_FOR_UMAP} synapse PCs")

umap_coords = plot_synapse_umap(
    Z_pcs, adata_full,
    n_pcs=N_PCS_FOR_UMAP,
    colors=('condition', 'time', 'cell_system'),
    n_pc_panels=5,
    seed=SEED,
    save_path=RESULTS_DIR / 'synapse_umap.png',
)


In [ ]:
# Cell 7d — W (marker) loadings for Z factors most interacting with synapse_PC2
# (factors 5 and 9 identified from the combined heatmap in Cell 6)
FACTORS_OF_INTEREST = [5, 9]
TOP_N_W = 15

for f in FACTORS_OF_INTEREST:
    if f >= W_df.shape[1]:
        print(f"Skip Z[{f}] - only {W_df.shape[1]} factors fit")
        continue
    w_vec   = W_df.iloc[:, f]
    top_idx = w_vec.abs().sort_values(ascending=False).head(TOP_N_W).index
    signed  = w_vec.loc[top_idx]
    col = f'Z[{f}]'
    if 'synapse_PC2' in syn_z_corr.index and col in syn_z_corr.columns:
        rho  = syn_z_corr.loc['synapse_PC2', col]
        padj = syn_z_padj.loc['synapse_PC2', col]
        header = f"Z[{f}] vs synapse_PC2  rho={rho:+.3f}  padj={padj:.2g}"
    else:
        header = f"Z[{f}]"
    print(f"\n--- {header}  (top {TOP_N_W} signed W loadings) ---")
    for feat, val in signed.items():
        sign = '+' if val >= 0 else '-'
        print(f"  {sign} {abs(val):.3f}   {feat}")


In [ ]:
# Cell 7b — Per-PC violins across (time × condition) × system — all top PCs
import importlib, factor_utils; importlib.reload(factor_utils)
from factor_utils import plot_synapse_pc_violins_by_system

plot_synapse_pc_violins_by_system(
    synapse_df, adata_full,
    pcs=tuple(f'synapse_PC{i + 1}' for i in range(N_TOP_PCS)),
    save_dir=RESULTS_DIR,
)


In [ ]:
# Cell 9 — Deep-dive on best Z factor for synapse_PC1
import importlib, factor_utils; importlib.reload(factor_utils)
from factor_utils import plot_factor_synapse_deepdive

best_f_name = syn_z_corr.loc['synapse_PC1'].abs().idxmax()   # e.g. 'Z[3]'
best_f = int(best_f_name.split('[')[1].split(']')[0])
rho  = syn_z_corr.loc['synapse_PC1', best_f_name]
padj = syn_z_padj.loc['synapse_PC1', best_f_name]
print(f"Best Z factor for synapse_PC1: {best_f_name}  ρ={rho:.3f}  padj={padj:.3g}")

plot_factor_synapse_deepdive(
    best_f=best_f, Z_arr=Z_arr,
    synapse_df_aligned=synapse_df_aligned,
    W_df=W_df, pc_loadings=loadings, adata=adata,
    pc_name='synapse_PC1', n_top=15,
    save_path=RESULTS_DIR / f'synapse_topZ_Z{best_f}_deepdive.png',
)
